# Second-order time stepping

The previous lessons are a great foundation in numerical methods for studying dynamical systems governed by ordinary differential equations.
You learned to apply Euler's method and studied its rate of convergence using numerical experiments. An exercise **on paper** in [Lesson 2](./02-oscillation.ipynb) using Taylor expansions showed that convergence to be first order. We found numerical evidence of this behavior in [Lesson 3](./03-full-model.ipynb) using the full nonlinear phugoid model.

Euler's method, however, showed a clear weakness for the undamped oscillator in Lesson 2: every step artificially increased its oscillation amplitude. You can understand this graphically. Each Euler step advances a variable (for example, position) to the next time interval using:

$$
x\left(t_i+\Delta t\right) \approx x\left(t_i\right)+x^{\prime}\left(t_i\right) \Delta t
$$

Recall that the derivative of a function corresponds to the slope of the tangent at a point. Euler's method uses the slope at the initial point in an interval, and advances the numerical position with that initial velocity. In the sketch below we exaggerate the result: the Euler estimate overshoots the exact solution and the amplitude of the oscillation grows unphysically, adding a small amount of energy with every step.

```{figure} ./figures/euler-overshoot.png
:label: fig-euler-overshoot
:alt: A line plot of position vs. time in oscillatory motion, with slopes at two points overshooting the curve
:width: 500px
:align: center

Sketch of two Euler steps approximating a curved time-dependent function.
```

First-order methods in general require very small time steps to achieve acceptable accuracy, and thus they are rarely used for real-world dynamical calculations. 
We now investigate a method with a higher order of accuracy. Among the most popular higher-order methods are the _Runge-Kutta methods_, developed around 1900: more than 100 years after Euler published his book containing the method now named after him.

## Paper-airplane challenge

Throughout this lesson, we use this motivating challenge: how far can you make a paper airplane fly?

Once released, a paper airplane has no propulsion or active control; its flight depends strongly on the launch (angle and speed) and aerodynamic response. If we use computation to recommend a launch, we need to distinguish a genuine improvement from a change caused by time-stepping error.

Our engineering question is:

> Find a competitive launch within specified bounds, then determine which method supports its predicted range to the required accuracy with less computational work.

You will use the following common problem data:

- Model parameters: $C_L=1$, $C_D=0.2$ (so $L/D=5$), $v_t=4.9\ \mathrm{m/s}$, and $g=9.81\ \mathrm{m/s^2}$. The lift-to-drag ratio is motivated by measurements reported by @feng2009.
- Release point: $x_0=0$ and $y_0=h=2\ \mathrm{m}$. The release height is fixed.
- Launch-search bounds: $4\leq v_0\leq12\ \mathrm{m/s}$ and $-30^\circ\leq\theta_0\leq30^\circ$. Convert angles to radians before using the model. These bounds define a model exercise, not experimentally validated launch limits.
- Required numerical range accuracy: $e_R=0.01\ \mathrm{m}$ (1 cm). This is a target for the numerical calculation within the model, not a claim of centimeter accuracy for a real paper airplane.

Define range as the net horizontal displacement $R=x_{\mathrm{ground}}-x_0$ at the **first** crossing from positive altitude to nonpositive altitude. A run that has not reached the ground by the common time limit of 15 s, or encounters an invalid state, does not supply a usable range. This challenge is adapted from the computational phugoid exercise of @simanca2002.

The investigation has two stages:

1. **Controlled comparison:** keep the launch fixed at $v_0=6.5\ \mathrm{m/s}$ and $\theta_0=-0.1\ \mathrm{rad}$. Compare Forward Euler and explicit midpoint RK2 using the same ground-crossing procedure, then determine the work each needs to support the range-accuracy target. The [comparison brief](#paper-airplane-controlled-comparison) follows the convergence study.
2. **Agent-supported launch investigation:** search within the stated bounds, retain several leading candidates, and refine both their time steps and the sampling of nearby launch conditions. Then compare the methods at the same range accuracy. A competitive launch is supported relative to the tested alternatives; it is not a proven global optimum. The [agent activity](#paper-airplane-agent-search) follows the controlled comparison.

Work through these stages in order. First derive and inspect the midpoint update, then reconstruct and verify the shared touchdown evaluator and complete the controlled comparison. In Stage 2, an AI agent may write the repetitive search code into your notebook, but you will inspect and execute each new cell yourself.

## Runge–Kutta methods

A method's order of convergence describes how the accumulated, or **global**, discretization error changes as we refine the time step over a fixed time interval. For a sufficiently smooth solution, a convergent method of order $p$ has global error ${\mathcal O}(\Delta t^p)$. When the leading discretization-error term dominates, we expect approximately

$$
e \approx C(\Delta t)^p,
$$

where $e$ measures the global error and $C$ depends on the problem, method, and time interval, but not on $\Delta t$. First-order error ($p=1$) scales linearly with the step size; second-order error ($p=2$) scales quadratically. These are small-step trends, not a guarantee that a higher-order method has a smaller error at every chosen step size.

One idea for improving on Euler's method is to estimate the derivative at an intermediate point, like the **midpoint**, which results in the so-called *explicit midpoint method* or *modified Euler method*. The scheme has two steps and is written as:

$$
\label{eq-rk2-midpoint-system}
\begin{aligned}
u_{n+1/2}   & = u_n + \frac{\Delta t}{2} f(u_n) \\
u_{n+1} & = u_n + \Delta t \,\, f(u_{n+1/2})
\end{aligned}
$$

Notice that this step evaluates the right-hand side, $f(u)$, twice: at the current state and at a predicted midpoint state. The notation $u_{n+1/2}$ represents an estimate of the state at $t_n + \frac{\Delta t}{2}$. All Runge–Kutta methods use such intermediate evaluations, called **stages**. Their order depends on how the stage states are constructed and how the derivative evaluations are combined—not simply on how many evaluations are performed.

:::{warning .simple .dropdown icon=false open=false} On paper — explain the midpoint update

First consider a scalar autonomous equation $u'=f(u)$, with a sufficiently smooth $f$. Start a single step from the exact value $u_n=u(t_n)$ and use

$$
\label{eq-midpoint-taylor-preparation}
u(t_n+\Delta t)=u_n+\Delta t\,u'_n+\frac{\Delta t^2}{2}u''_n+{\mathcal O}(\Delta t^3).
$$

1. Apply the chain rule to write $u''_n=f'(u_n)f(u_n)$. Expand $f(u_n+\tfrac12\Delta t\,f(u_n))$ and substitute it into [Equation %s](#eq-rk2-midpoint-system). Show that the numerical update matches [Equation %s](#eq-midpoint-taylor-preparation) through the terms in $\Delta t^2$.
2. Explain why the midpoint state is a prediction, and why the final update must start from $u_n$, not from that prediction.
3. Identify the ${\mathcal O}(\Delta t^3)$ one-step defect and explain why, under the smoothness and stability assumptions for convergence over a fixed interval, the accumulated error is second order.

When you reach `rk2_step()` below, check that both stages use complete state vectors. The scalar derivation motivates the construction; the implementation must advance all four components together.
:::

Explicit midpoint is a second-order Runge–Kutta method (RK2). Its higher order does not eliminate all of Euler's limitations. For the undamped oscillator in Lesson 2, sufficiently small time steps give less artificial amplitude growth than Forward Euler, but the amplitude still grows for any fixed, nonzero step size. Convergence as the step size shrinks over a fixed time interval is different from stable long-time behavior at a fixed step size.

An interesting historical connection to our phugoid problem: Carl Runge's daughter Iris—an accomplished applied mathematician in her own right—worked assiduously over the summer of 1909 to translate Lanchester's _"Aerodonetics."_ She also reproduced his graphical method to draw the phugoid curves [@tobies2012, p. 73].

## Phugoid model with second-order RK

Let's begin the paper-airplane investigation by computing a baseline flight under the full phugoid model using both Forward Euler and second-order Runge–Kutta. We will use the same launch conditions for both methods and examine the horizontal distance traveled before the airplane touches the ground.

We will first compare the methods over a common interval while the airplane is still aloft. That fixed-time study checks the behavior of RK2, but it cannot yet justify a touchdown range or a time step for the engineering challenge. Afterward, we will build and verify the event calculation needed to measure range.

:::{warning .simple .dropdown icon=false open=false} In your notebook

Create a new notebook for your work and keep this published lesson open as a worked reference. Reconstruct the midpoint update, the fixed-time refinement study, and the touchdown evaluator in your own notebook, including the direct first draft of the evaluator and the failure that motivates its checks. Type the numerical updates and event logic yourself so that you can connect each line to the equations; copying mechanical details such as imports, module-download code, and plot labels is fine.

Pause at each checkpoint. Record what you expect before running a cell, inspect the reported differences and statuses, and keep the steady-glide result as evidence that your reconstructed event calculation works. Consult [Reconstruct a lesson](../../appendices/notebook-workflow.md#notebook-reconstruct) for the general workflow.
:::

Start by importing NumPy and Matplotlib with the aliases used in the previous lessons. We also set the font family and size through Matplotlib's `rcParams` dictionary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

Use the challenge's lift-to-drag ratio $L/D=5.0$ and trim speed of $4.9\ \mathrm{m/s}$. _What do you think will happen if you make $L/D$ higher?_

In [ ]:
# Model parameters.
g = 9.81         # gravitational acceleration (m/s**2)
v_t = 4.9        # trim speed (m/s)
C_D = 1.0 / 5.0  # drag coefficient
C_L = 1.0        # lift coefficient

# Initial conditions.
v_0 = 6.5       # initial speed, above the trim speed (m/s)
theta_0 = -0.1  # trajectory angle (rad)
x_0 = 0.0       # horizontal position (m)
y_0 = 2.0       # altitude (m)

The initial speed is a little higher than the trim speed, the launch angle is negative, and the release height is 2 meters. We will use the same initial state and parameters for both numerical methods.

### Reuse functions from a Python module

We have already written and inspected these three functions in [Lesson 3](./03-full-model.ipynb):

- `rhs_full_phugoid()` gives the four model derivatives.
- `euler_step()` advances the complete state by one Forward Euler step, in the `*args` form.
- `discrete_l1_difference()` compares histories on nested time grids.

These functions are now reused often enough to save in a separate file. A **module** is an ordinary Python source file with the extension `.py`. To make one, create a plain-text file, copy the function definitions into it, and include the imports those definitions need—these functions only need `import numpy as np`. Keep parameter choices, integration loops, plots, and notebook-only commands in the notebook.

The course provides these definitions in a file named `phugoid.py`. [Read the module source](https://github.com/numerical-mooc/practical-numerical-methods/blob/main/src/phugoid.py): its three functions are the ones already presented in previous lessons. We can download this file directly using `urlretrieve()` from Python's standard library; add the code below to your notebook and execute it.

In [ ]:
from urllib.request import urlretrieve

url = (
    'https://raw.githubusercontent.com/'
    'numerical-mooc/practical-numerical-methods/main/'
    'src/phugoid.py'
)
fname = 'phugoid.py'
urlretrieve(url, fname)

Here, `url` points to the raw Python file, and `fname` names the local copy. The file is saved in the kernel's current working directory, normally the folder containing your notebook. Keep `phugoid.py` alongside your working notebook. You only need to download it once; rerunning the download overwrites that file, so save a separate copy of any local edits first.

Downloading saves the file; **importing** makes its functions available in the notebook. In the import statement, use the filename without `.py`. Python executes a module's top-level statements when it first loads it, so you should only import code from sources you trust and have inspected. Our file imports NumPy and defines functions; it does not run a simulation.

Functions in the module do not inherit notebook variables, so we continue to pass the state and model parameters explicitly. The difference function requires **both** time-step sizes to check grid nesting and endpoint alignment.

If you edit the local module, restart the kernel and rerun your imports and calculations, skipping the download cell to preserve your edits. Rerunning an import alone does not reload an already imported module. For more detail, see the [Python modules tutorial](https://docs.python.org/3/tutorial/modules.html).

In [ ]:
from phugoid import (
    discrete_l1_difference,
    euler_step,
    rhs_full_phugoid,
)

### Define the RK2 step

The reused functions now come from the module, but the new numerical method stays visible here and in your notebook, as it is the first time you encounter it. Define `rk2_step()` to implement the modified Euler method in [Equation %s](#eq-rk2-midpoint-system), also known as second-order Runge–Kutta or RK2. The time loop will call this function once per step.

In [ ]:
def rk2_step(u, f, dt, *args):
    '''Return the next state using the second-order Runge–Kutta method.

    Parameters
    ----------
    u : np.ndarray
        State at the current time
        as a 1D array of floats.
    f : function
        Function to compute the right-hand side of the system.
    dt : float
        Time-step size.
    *args
        Additional positional arguments passed to f.

    Returns
    -------
    u_new : np.ndarray
        The solution at the next time step
        as a 1D array of floats.
    '''
    u_star = u + 0.5 * dt * f(u, *args)
    u_new = u + dt * f(u_star, *args)
    return u_new

### Compare on a common time interval

Begin with an illustrative time step of $\Delta t=0.01\ \mathrm{s}$. We have not yet shown that this step is accurate enough for touchdown range, so do not treat it as an engineering recommendation. Integrate only to $T=2\ \mathrm{s}$, when the baseline flight is still above ground, and compare the two approximations on exactly the same time grid.

As in [Lesson 3](./03-full-model.ipynb), the variable `num_steps` counts updates; each history has `num_steps + 1` rows to include the initial state. Both arrays have four columns ordered as `[v, theta, x, y]`.

In [ ]:
T_trajectory = 2.0  # common pre-impact interval (s)
dt = 0.01  # time-step size (s)
num_steps = int(round(T_trajectory / dt))

# Store the state at every time point, including the initial state.
u_euler = np.empty((num_steps + 1, 4))
u_rk2 = np.empty((num_steps + 1, 4))
u_euler[0] = np.array([v_0, theta_0, x_0, y_0])
u_rk2[0] = np.array([v_0, theta_0, x_0, y_0])

# Advance both methods over the same time grid.
for n in range(num_steps):
    u_euler[n + 1] = euler_step(
        u_euler[n], rhs_full_phugoid, dt, C_L, C_D, g, v_t
    )
    u_rk2[n + 1] = rk2_step(
        u_rk2[n], rhs_full_phugoid, dt, C_L, C_D, g, v_t
    )

Extract time, horizontal position, and altitude for the exploratory plot. The final altitudes will also confirm that this entire comparison ends before impact.

In [ ]:
# Extract the common time grid and both position histories.
t_trajectory = np.linspace(0.0, T_trajectory, num_steps + 1)
x_euler = u_euler[:, 2]
y_euler = u_euler[:, 3]
x_rk2 = u_rk2[:, 2]
y_rk2 = u_rk2[:, 3]
print(f'Final Euler altitude: {y_euler[-1]:.3f} m')
print(f'Final RK2 altitude:   {y_rk2[-1]:.3f} m')

### Plot for exploration, not an accuracy test

A trajectory plot is a useful first diagnostic: it can expose a wrong sign, a discontinuity, or an implausible path. It cannot show that a range is accurate to 1 cm. Two curves can overlap visually while differing by more than the required tolerance.

For now, inspect the two paths and their separation. This is exploratory evidence, not a verdict.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
for ax in axes:
    ax.grid()
    ax.set_xlabel('Horizontal position, x (m)')
    ax.set_ylabel('Altitude, y (m)')

# Plot both approximations over the common pre-impact interval.
axes[0].plot(x_euler, y_euler, label='Euler')
axes[0].plot(x_rk2, y_rk2, label='RK2')
axes[0].legend()

# Plot their horizontal separation on the same time grid.
axes[1].plot(t_trajectory, x_rk2 - x_euler)
axes[1].set_xlabel('Time, t (s)')
axes[1].set_ylabel(r'$x_{RK2} - x_{Euler}$ (m)')
fig.tight_layout()

## Fixed-time trajectory convergence

Just like in [Lesson 3](./03-full-model.ipynb), we want to check whether RK2 exhibits its expected convergence rate on this problem. Every history below ends at the same pre-impact time, $T=2\ \mathrm{s}$, so values at matching indices describe the same physical times.

The `for` loop computes the solution on several nested time grids, with the coarsest and finest step sizes differing by a factor of 100. We then compare horizontal position, a single quantity with units of meters. Mixing all four state columns would combine speed, angle, horizontal position, and altitude in one number with no coherent physical units.

In [ ]:
# Set the time-step sizes to investigate.
dt_values = [0.1, 0.05, 0.01, 0.005, 0.001]
u_histories = []

for dt_trial in dt_values:
    num_steps_trial = int(round(T_trajectory / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = rk2_step(
            u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_histories.append(u_trial)

For the general nonlinear trajectory computed here, we do not have an exact reference solution available. The finest-grid history is a numerical reference, not an exact solution, so the differences below are not exact errors. They are evidence about refinement behavior over this fixed interval.

Once those runs are complete, compare each horizontal-position history with the finest-grid history. Pass both time-step sizes to the imported `discrete_l1_difference()` function so it can align the grids.

In [ ]:
# Compute the differences in horizontal position.
difference_values = []
for u_trial, dt_trial in zip(u_histories, dt_values, strict=True):
    difference = discrete_l1_difference(
        u_trial[:, 2], u_histories[-1][:, 2],
        dt_trial, dt_values[-1],
    )
    difference_values.append(difference)

Plot the differences against time-step size on logarithmic axes, as in the previous lessons.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.set_title(r'$L_1$ difference vs. time-step size')
ax.set_xlabel(r'$\Delta t$ (s)')
ax.set_ylabel(r'$L_1$ difference in $x$')
ax.grid()
ax.loglog(
    dt_values[:-1], difference_values[:-1],
    color='tab:blue', linestyle='--', marker='o',
)
ax.set_aspect('equal', adjustable='box')
fig.tight_layout()

The decreasing differences suggest convergence, but their size is measured relative to a numerical reference. In [Lesson 3](./03-full-model.ipynb), the observed order for Euler's method was close to 1. For RK2, the expectation is $p\approx2$ once the leading discretization-error term dominates. Let us test that expectation.

To compute the observed order of convergence, we use three grid resolutions that are refined at a constant rate, in this case $r=2$.

In [ ]:
refinement_ratio = 2
dt_fine = 0.001
dt_order = [
    dt_fine,
    refinement_ratio * dt_fine,
    refinement_ratio**2 * dt_fine,
]
u_order = []

for dt_trial in dt_order:
    num_steps_trial = int(round(T_trajectory / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = rk2_step(
            u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_order.append(u_trial)

# Compare horizontal position on the three grids.
difference_coarse_medium = discrete_l1_difference(
    u_order[2][:, 2], u_order[1][:, 2],
    dt_order[2], dt_order[1],
)
difference_medium_fine = discrete_l1_difference(
    u_order[1][:, 2], u_order[0][:, 2],
    dt_order[1], dt_order[0],
)
difference_ratio = difference_coarse_medium / difference_medium_fine
observed_order = (
    np.log(difference_ratio) / np.log(refinement_ratio)
)

print(f'Coarse–medium difference: {difference_coarse_medium:.6e} m s')
print(f'Medium–fine difference:   {difference_medium_fine:.6e} m s')
print(f'Difference ratio:         {difference_ratio:.3f}')
print(f'Observed order:           p = {observed_order:.3f}')

An observed order close to $2$ is consistent with the expected second-order behavior for this grid family. In the regime where the leading error is proportional to $\Delta t^2$, halving the step size reduces that error to approximately one quarter of its previous size. The raw differences and their ratio make that conclusion inspectable rather than reporting only the final value of $p$.

This study concerns horizontal-position histories over a fixed, airborne interval. It does **not** verify touchdown range or justify $\Delta t=0.01\ \mathrm{s}$ for the challenge. Touchdown occurs at a step-dependent time, and locating it introduces a separate event-calculation error. We need to define and test that measurement before studying range convergence.

## Locate touchdown and evaluate range

Touchdown is an **event**: it happens when the altitude first crosses the ground, not necessarily at one of our stored times. Starting from positive altitude, one numerical step brackets touchdown when

$$
y_n>0,\qquad y_{n+1}\leq0.
$$

Assume the state changes linearly between those two computed endpoints. Let $\alpha$ be the fraction of the step needed to reach $y=0$. Linear interpolation gives

$$
\label{eq-touchdown-linear-interpolation}
\alpha=\frac{y_n}{y_n-y_{n+1}},\quad
t_{\mathrm{g}}=t_n+\alpha(t_{n+1}-t_n),\quad
u_{\mathrm{g}}=u_n+\alpha(u_{n+1}-u_n).
$$

Because the two altitudes bracket zero, $0<\alpha\leq1$. The horizontal component of $u_{\mathrm{g}}$ supplies the interpolated touchdown position, and the net range is $R=x_{\mathrm{g}}-x_0$. We retain the two computed endpoints as the crossing bracket, but stop immediately: no later below-ground states are calculated.

A usable result also needs an explicit completion status. We will report `touchdown`, `time_limit`, or `invalid_state`; the third arises when a calculation breaks, and we will see below why it needs a status of its own. Failed or unfinished runs receive no range.

:::{warning .simple .dropdown icon=false open=false} On paper — interpolate one crossing

A step from $t_n=2.00\ \mathrm{s}$ to $t_{n+1}=2.01\ \mathrm{s}$ takes the altitude from $y_n=0.3\ \mathrm{m}$ to $y_{n+1}=-0.1\ \mathrm{m}$, the horizontal position from $x_n=5.00\ \mathrm{m}$ to $x_{n+1}=5.06\ \mathrm{m}$, and the speed from $6.0$ to $6.1\ \mathrm{m/s}$.

1. Compute $\alpha$ from [Equation %s](#eq-touchdown-linear-interpolation) and confirm that $0<\alpha\leq1$.
2. Compute the touchdown time $t_{\mathrm{g}}$, the horizontal position $x_{\mathrm{g}}$, and the speed at touchdown.
3. Explain why the interpolated altitude must be exactly zero.

Keep these values: they become the expected values in a test below.
:::


### Write the interpolation as a function

The crossing formula is small enough to be its own function, so that we can test it by hand before it is buried inside a loop. `locate_touchdown()` receives the two times and states that bracket the crossing and returns the interpolated touchdown time and state.


In [ ]:
def locate_touchdown(t, u, t_next, u_next):
    '''Interpolate the ground crossing inside one bracketing step.

    Parameters
    ----------
    t, t_next : float
        Times at the start and end of the bracketing step.
    u, u_next : np.ndarray
        States at those times, with u[3] > 0 and u_next[3] <= 0.

    Returns
    -------
    t_touchdown : float
        Interpolated time of the crossing.
    u_touchdown : np.ndarray
        Interpolated state at the crossing.
    '''
    alpha = u[3] / (u[3] - u_next[3])
    t_touchdown = t + alpha * (t_next - t)
    u_touchdown = u + alpha * (u_next - u)
    return t_touchdown, u_touchdown


Check the function against your hand calculation before using it. The bracket below uses the values from the **On paper** exercise; the expected values come from that calculation, not from the code.


In [ ]:
u_above = np.array([6.0, -0.2, 5.0, 0.3])
u_below = np.array([6.1, -0.2, 5.06, -0.1])
t_check, u_check = locate_touchdown(2.0, u_above, 2.01, u_below)

np.testing.assert_allclose(t_check, 2.0075)
np.testing.assert_allclose(u_check, [6.075, -0.2, 5.045, 0.0])


### Start with a direct loop

Before adding any checks, write the range calculation directly: take a step, test for a crossing, interpolate, and stop. If the flight is still aloft when the steps run out, report that instead. A `for` loop over `int(np.floor(time_limit / dt))` takes only complete steps within the safety cap; ending less than one step early is harmless here. The function returns a status and a range, and the status `time_limit` means that no crossing was found.


In [ ]:
def range_first_draft(step_function, u_0, f, dt, time_limit, *args):
    '''Return a status and range from a loop with no validity checks.'''
    u = np.asarray(u_0, dtype=float)
    x_initial = u[2]
    for n in range(int(np.floor(time_limit / dt))):
        u_next = step_function(u, f, dt, *args)
        if u_next[3] <= 0.0:
            t_touchdown, u_touchdown = locate_touchdown(
                n * dt, u, (n + 1) * dt, u_next
            )
            return 'touchdown', u_touchdown[2] - x_initial
        u = u_next
    return 'time_limit', None


Apply this function to the baseline launch, then to a launch with zero initial speed. The model contains $g/v$, so zero speed is outside its domain. Predict what the draft will report in the second case before running the cell.


In [ ]:
u_baseline = np.array([v_0, theta_0, x_0, y_0])
print(range_first_draft(
    euler_step, u_baseline, rhs_full_phugoid, 0.01, 15.0, C_L, C_D, g, v_t
))

u_zero_speed = np.array([0.0, theta_0, x_0, y_0])
print(range_first_draft(
    euler_step, u_zero_speed, rhs_full_phugoid, 0.01, 15.0, C_L, C_D, g, v_t
))


### A silent wrong answer

The first result is the interpolated baseline range. The second is wrong in a way that is easy to miss. NumPy printed a warning about division by zero, but a warning does not stop a calculation. The first step produced an infinite trajectory angle, and the following step turned the state into `nan`, which stands for "not a number." Every comparison with `nan` is `False`, including `u_next[3] <= 0.0`, so the crossing test never fires and the loop runs to the time limit. The draft function then reports that the airplane never landed, when in fact the calculation broke on its first step. Confirm the behavior of the comparison:


In [ ]:
print(np.nan <= 0.0, np.nan > 0.0, np.isfinite(np.nan))


A result that looks like a legitimate outcome, `time_limit`, but comes from a broken calculation is a type of failure that needs to be prevented with defensive checks. The remedy is a validity test on every completed step, placed **before** the crossing test, so that a broken state is reported as such rather than silently compared. A state is valid when every entry is finite and the speed is positive: negative speed is outside the state convention, and zero speed is outside the model's domain. A nonpositive altitude reached from a positive altitude is the event, not an invalid state.

We can write this test as a small function and check it on a valid state, the zero-speed state, and a state containing `nan`.


In [ ]:
def state_is_valid(u):
    '''Return True when every entry is finite and the speed is positive.'''
    return np.all(np.isfinite(u)) and u[0] > 0.0


assert state_is_valid(u_baseline)
assert not state_is_valid(u_zero_speed)
assert not state_is_valid(np.array([np.nan, theta_0, x_0, y_0]))


:::{note} Python refresher
:icon: false
`assert not` verifies that an invalid state is correctly rejected. For example,`u_zero_speed` has speed `u[0] == 0`, while `state_is_valid()` requires positive speed. Therefore:
```python
state_is_valid(u_zero_speed)  # False
not False                     # True
```
Since the asserted expression is `True`, the test passes. If the function mistakenly returned `True` for zero speed or `NaN`, Python would raise an `AssertionError`. We are testing the function’s invalid-input behavior.
:::

### Map the possible outcomes

Before any more coding, you should understand the possible outcomes. The complete evaluator has four jobs:

1. **Check the setup.** A nonpositive time step, an initial state with the wrong number of entries, or a nonpositive initial altitude is a caller error: no crossing from positive altitude could ever be found, and the draft function above would silently report `time_limit`. The full function raises `ValueError` rather than starting a calculation with a broken setup.
2. **Advance and count steps.** The evaluator counts accepted completed steps, including the step that brackets touchdown. A step that first exposes an invalid state is rejected and not counted; failed runs do not enter the work comparison. For successful runs, multiplying by one right-hand-side evaluation per Euler step or two per midpoint step gives the computational work.
3. **Recognize a failed flight calculation.** A completed step whose state is non-finite or has nonpositive speed produces the status `invalid_state` and no range. The check inspects completed steps only; a stricter evaluator would also inspect RK2's intermediate stage.
4. **Recognize completion.** The first positive-to-nonpositive altitude bracket produces `touchdown`; using up the steps without such a bracket produces `time_limit`. Only touchdown supplies a range.

Here is the same control flow as pseudocode. Read this outline first, then find each part in the Python function below.

```text
check the setup

for each step up to the time limit:
    take one numerical step
    if the new state is invalid:
        stop with invalid_state
    store the new state
    if altitude crossed zero:
        interpolate touchdown
        stop with touchdown

report time_limit
```

At the start of the function, `np.asarray(u_0, dtype=float)` normalizes a list or array into floating-point NumPy data. We call that working state `u` because it changes as the integration advances, while `u_0` names the original initial condition. The initial horizontal position is saved separately so the final range can always be measured from the launch point.


:::{note} Python refresher — passing functions and stopping a loop early
:icon: false

Python functions are values, so a function name without parentheses can be passed into another function. The parameter `step_function` below can therefore refer to either `euler_step` or `rk2_step`; the shared loop does not need separate event logic for the two methods. The syntax `*args` collects extra positional arguments into a tuple, and `step_function(u, f, dt, *args)` unpacks that tuple again when making the call.

The `for` loop runs over `range(max_steps)`, and `break` leaves the loop as soon as touchdown or failure occurs. `np.floor()` rounds the quotient down before `int()` converts it to an integer, so the loop never steps beyond the safety cap. Because the loop may stop early, the times and states are collected in Python lists and converted to arrays at the end, as in the trammel tracer of [Lesson 1](./01-theory.ipynb). The function returns one dictionary rather than a long tuple; the refresher after the code explains how to read it.
:::


In [ ]:
def integrate_until_touchdown(
    step_function, u_0, f, dt, time_limit, *args
):
    '''Integrate until touchdown, the time limit, or an invalid state.'''
    if dt <= 0.0:
        raise ValueError('dt must be positive.')

    u = np.asarray(u_0, dtype=float)
    if u.shape != (4,) or u[3] <= 0.0:
        raise ValueError(
            'u_0 must contain [v, theta, x, y] with positive altitude.'
        )
    x_initial = u[2]

    max_steps = int(np.floor(time_limit / dt))
    times = [0.0]
    states = [u.copy()]
    status = 'time_limit'
    touchdown_time = None
    touchdown_state = None
    range_value = None

    for n in range(max_steps):
        u_next = step_function(u, f, dt, *args)

        if not state_is_valid(u_next):
            status = 'invalid_state'
            break

        t_next = (n + 1) * dt
        times.append(t_next)
        states.append(u_next)

        if u_next[3] <= 0.0:
            touchdown_time, touchdown_state = locate_touchdown(
                n * dt, u, t_next, u_next
            )
            range_value = touchdown_state[2] - x_initial
            status = 'touchdown'
            break

        u = u_next

    return {
        'status': status,
        'times': np.asarray(times),
        'states': np.asarray(states),
        'touchdown_time': touchdown_time,
        'touchdown_state': touchdown_state,
        'range': range_value,
        'steps': len(times) - 1,
    }


:::{note} Python refresher — dictionaries and `None`
:icon: false

A dictionary groups related values under descriptive keys. The expression `result['status']` retrieves the value stored under `status`, which is easier to read here than remembering positions in a long tuple. The special value `None` means that no value is available, and `is None` checks for that exact marker. Thus a timed-out or invalid run can still return its status, history, and step count while `result['range']` remains `None`; it is not replaced by zero or by the last stored horizontal position. Later, `.items()` lets a loop retrieve each dictionary key and its associated value together.
:::

### Check the non-success outcomes

A failure path deserves a test just as much as a successful calculation. Use a deliberately short time limit to produce an unfinished run, then supply the zero-speed launch that fooled the first draft of the evaluator. In both cases, inspect the status and confirm that range is unavailable. The zero-speed run now rejects its first attempted step instead of running to the time limit; its reported `steps` value is therefore zero.


In [ ]:
unfinished_result = integrate_until_touchdown(
    euler_step, u_baseline, rhs_full_phugoid,
    0.01, 0.1, C_L, C_D, g, v_t,
)
invalid_result = integrate_until_touchdown(
    euler_step, u_zero_speed, rhs_full_phugoid,
    0.01, 15.0, C_L, C_D, g, v_t,
)

for result in (unfinished_result, invalid_result):
    print(
        f'{result["status"]}: range = {result["range"]}; '
        f'{result["steps"]} accepted steps'
    )

assert unfinished_result['status'] == 'time_limit'
assert unfinished_result['range'] is None
assert unfinished_result['steps'] == 10
assert invalid_result['status'] == 'invalid_state'
assert invalid_result['range'] is None
assert invalid_result['steps'] == 0


The initial-state guard also deserves a test. The cell below deliberately passes a release point below ground (`-1.0`) and confirms that the function refuses to start. It is a small test asking: _“Does the function refuse to simulate an aircraft that already starts below ground?”_ The draft evaluator would have treated the first update as touchdown because its altitude was already nonpositive. With both endpoints below ground, `locate_touchdown()` would extrapolate to a fictitious crossing outside the step rather than interpolate a real bracket.

:::{note} Python refresher — `try` and `except`
:icon: false

A `raise` statement stops the program with an error unless something catches it. The `try` block runs code that may raise; if it raises the named error type, execution jumps to the `except` block, where the name `error` refers to the error object and its message. This is the standard way to test that a function rejects invalid input without stopping the calculation. The final `assert` would fail if the guard were removed.
:::


In [ ]:
u_below_ground = np.array([v_0, theta_0, x_0, -1.0])
guard_triggered = False
try:
    integrate_until_touchdown(
        euler_step, u_below_ground, rhs_full_phugoid,
        0.01, 15.0, C_L, C_D, g, v_t,
    )
except ValueError as error:
    guard_triggered = True
    print('Rejected:', error)

assert guard_triggered


The invalid input is created with `-1.0` on the last element of `u_below_ground`. The test begins by assuming the guard has not run:
```python
guard_triggered = False
```
Then it tries to call the function:
```python
try:
    integrate_until_touchdown(...)
```
Inside the function, this condition detects the negative altitude:
```python
if u.shape != (4,) or u[3] <= 0.0:
    raise ValueError(...)
```
Normally that would stop the cell. However, the `except` block catches that particular error, and `guard_triggered = True` records that the expected rejection occurred. The final `assert` checks that the function really did reject the input.

The complete flow is:
```text
Negative altitude supplied
        ↓
Function raises ValueError
        ↓
except catches it
        ↓
guard_triggered becomes True
        ↓
assert passes
```
Reread this little test and explanation as many times as you need. Check your understanding by answering: what would happen if the altitude check were accidentally removed?

### Verify the evaluator with steady glide

The general nonlinear flight lacks an exact solution we can use to verify the evaluator, but the model does have an exact steady-glide special case. **On paper**, set $v'=0$ and $\theta'=0$ in the [model equations](./03-full-model.ipynb#eq-full-phugoid-dynamics). With $\eta=C_D/C_L$, show that the descending equilibrium has

$$
\label{eq-paper-airplane-steady-glide-state}
\theta_s=-\arctan\eta,\qquad v_s=v_t\sqrt{\cos\theta_s}.
$$

The steady-glide speed $v_s$ is not exactly the trim-speed parameter $v_t$. Use the constant derivatives $x'=v_s\cos\theta_s$ and $y'=v_s\sin\theta_s$ to eliminate time and show that the range from height $h$ is

$$
\label{eq-paper-airplane-steady-glide-range}
R_s=-h\frac{\cos\theta_s}{\sin\theta_s}=h\frac{L}{D}=10\ \mathrm{m}.
$$

Use $[v_s,\theta_s,0,h]$ as a separate verification initial state, not as a replacement for the baseline launch. Because the vertical velocity is constant, the exact touchdown time is $t_{\mathrm{g}}=-h/(v_s\sin\theta_s)$. Both Euler and RK2 advance this straight-line motion exactly apart from floating-point effects.

The step sizes below place touchdown strictly between stored times. For every trial, require the `touchdown` status, confirm that the interpolated time lies inside the final bracket, and compare the range with [Equation %s](#eq-paper-airplane-steady-glide-range). This checks the event calculation; it does not establish accuracy for a curved trajectory or validate the physical model.

In [ ]:
eta = C_D / C_L
theta_steady = -np.arctan(eta)
v_steady = v_t * np.sqrt(np.cos(theta_steady))
u_steady = np.array([v_steady, theta_steady, 0.0, y_0])
exact_steady_time = -y_0 / (v_steady * np.sin(theta_steady))
exact_steady_range = y_0 * C_L / C_D
verification_dts = [0.5, 0.4, 0.25]

print('method      dt (s)   touchdown (s)   range (m)   error (m)')
for method_name, step_function in (
    ('Euler', euler_step), ('RK2', rk2_step)
):
    for dt_trial in verification_dts:
        result = integrate_until_touchdown(
            step_function, u_steady, rhs_full_phugoid,
            dt_trial, 15.0, C_L, C_D, g, v_t,
        )
        assert result['status'] == 'touchdown'
        range_error = abs(result['range'] - exact_steady_range)
        print(
            f'{method_name:<8} {dt_trial:8.2f} '
            f'{result["touchdown_time"]:15.9f} '
            f'{result["range"]:11.9f} {range_error:11.3e}'
        )

        assert (
            result['times'][-2]
            < result['touchdown_time']
            < result['times'][-1]
        )
        assert abs(result['touchdown_time'] - exact_steady_time) < 1e-10
        assert range_error < 1e-10

The assertions are deliberately demanding because this equilibrium produces straight-line motion: the time-stepping methods and the linear event model are exact for this case apart from floating-point rounding. Passing this test at several off-grid touchdown times gives us a focused check of the bracketing and interpolation logic. It does not tell us how small $\Delta t$ must be for the curved baseline flight; that requires range refinement.

(paper-airplane-controlled-comparison)=
## Compare methods at a required accuracy

This is Stage 1 of the paper-airplane challenge. Keep the baseline launch and all model parameters fixed. The question is not which method runs faster on the same grid, but which needs fewer right-hand-side evaluations to support a range accurate to 1 cm.

We now have a verified range evaluator with explicit completion statuses and a common event treatment for both methods. Our next goal is to apply it first to the baseline launch at the illustrative step size, then establish a numerical reference by refinement.


### Apply the evaluator to the baseline launch

Return to the baseline initial state and use the illustrative step size $\Delta t=0.01\ \mathrm{s}$. The next cell demonstrates the evaluator's interface and confirms that each successful run stops at its first crossing bracket. The printed ranges are interpolated, but they are still provisional numerical results: we have not yet established their errors by refinement.

In [ ]:
dt_touchdown = 0.01
time_limit = 15.0
rhs_calls_per_step = {'Euler': 1, 'RK2': 2}
flight_results = {}

for method_name, step_function in (
    ('Euler', euler_step), ('RK2', rk2_step)
):
    result = integrate_until_touchdown(
        step_function, u_baseline, rhs_full_phugoid,
        dt_touchdown, time_limit, C_L, C_D, g, v_t,
    )
    flight_results[method_name] = result
    work = result['steps'] * rhs_calls_per_step[method_name]

    if result['range'] is None:
        print(
            f'{method_name}: {result["status"]}; '
            f'range unavailable; {work} RHS evaluations'
        )
    else:
        print(
            f'{method_name}: {result["status"]}; '
            f't = {result["touchdown_time"]:.6f} s; '
            f'R = {result["range"]:.6f} m; '
            f'{work} RHS evaluations'
        )


In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
for method_name, result in flight_results.items():
    path = result['states'].copy()
    if result['status'] == 'touchdown':
        # Replace the below-ground bracket endpoint by the event point.
        path[-1] = result['touchdown_state']
    ax.plot(path[:, 2], path[:, 3], label=method_name)

ax.set_xlabel('Horizontal position, x (m)')
ax.set_ylabel('Altitude, y (m)')
ax.grid()
ax.legend()
fig.tight_layout()

For a successful run, `states` contains the two endpoints that bracket touchdown. The plotting cell keeps the last valid computed point but replaces the nonpositive endpoint with `touchdown_state`, so the displayed physical path ends at $y=0$. Work is counted in right-hand-side evaluations: the completed steps, including the bracketing step, multiplied by one evaluation per Euler step or two per midpoint step. The dictionary `rhs_calls_per_step` records those factors so that the conversion stays visible.

The two methods are not required to use the same time step in the final engineering comparison. Equal $\Delta t$ is a useful diagnostic, but it gives RK2 about twice the computational work while its computational accuracy is superior. Our primary question is thus: _what is the least RHS work each method needs to support the same range accuracy?_

### Establish a numerical range reference

The baseline ranges at one time step are provisional. Without an exact solution for the curved flight, the reference is a refined RK2 result, accepted when the range changes by less than 1 mm in each of two successive step halvings. That rule supports a working 1 mm allowance for reference uncertainty; it is evidence, not a rigorous error bound. The cell below applies the rule starting from $\Delta t=0.01\ \mathrm{s}$ and stops at the first step size that satisfies it, or reports insufficient evidence after eight halvings. It also adds up the work spent on the reference, which is reported separately from the comparison itself.


In [ ]:
reference_tolerance = 0.001  # allowance for reference uncertainty (m)
dt_reference = dt_touchdown
result = integrate_until_touchdown(
    rk2_step, u_baseline, rhs_full_phugoid,
    dt_reference, time_limit, C_L, C_D, g, v_t,
)
range_previous = result['range']
reference_work = result['steps'] * rhs_calls_per_step['RK2']
small_changes = 0
reference_accepted = False

print('dt (s)         range (m)    change (m)')
print(f'{dt_reference:<12.6f} {range_previous:12.6f}')
for halving in range(8):
    dt_reference = dt_reference / 2
    result = integrate_until_touchdown(
        rk2_step, u_baseline, rhs_full_phugoid,
        dt_reference, time_limit, C_L, C_D, g, v_t,
    )
    assert result['status'] == 'touchdown'
    reference_work += result['steps'] * rhs_calls_per_step['RK2']
    change = abs(result['range'] - range_previous)
    print(f'{dt_reference:<12.6f} {result["range"]:12.6f} {change:12.3e}')
    range_previous = result['range']

    if change < reference_tolerance:
        small_changes += 1
    else:
        small_changes = 0
    if small_changes == 2:
        reference_accepted = True
        break

assert reference_accepted, 'Insufficient evidence for a range reference.'
range_reference = range_previous
print(f'Reference range: {range_reference:.6f} m at dt = {dt_reference:g} s')
print(f'Work to establish the reference: {reference_work} RHS evaluations')


Two successive halvings changed the range by well under 1 mm, so the last step size in the table is the finest accepted RK2 step; it is the value you will enter as `dt_search` in Stage 2. In your notebook, also refine Euler and check that its ranges approach the same reference value. A first-order method converges more slowly, so expect several more halvings before the agreement is within 1 mm.


### Plan the accuracy-and-work comparison

The verified evaluator gives us trustworthy measurement logic, and the reference refinement shows how the accuracy of a range is established. Complete the comparison in your notebook using this plan:

1. **Establish a shared numerical reference.** Use the RK2 refinement rule demonstrated above: accept the reference when the range changes by less than $0.001\ \mathrm{m}$ in each of two successive step halvings, and check that refined Euler results approach the same value. Use the finest accepted result as $R_{\mathrm{ref}}$. These checks support a working 1 mm allowance for reference uncertainty; they are evidence, not a rigorous error bound. If that allowance is not credible, refine further before claiming the target accuracy.
2. **Measure error and work.** For each method, start the step-size ladder at $\Delta t=0.1\ \mathrm{s}$, the coarsest step of the fixed-time study, and halve it at least three times, refining further as needed. Starting coarse matters: if every tested step already meets the target, the least-work qualifying run is merely the coarsest run tried, and the comparison reveals nothing about where each method stops being accurate enough. Keep the same launch, event treatment, and 15 s time limit. Record the interpolated range, $|R-R_{\mathrm{ref}}|$, completion status, and number of right-hand-side evaluations, computed from the step count and `rhs_calls_per_step`. Report the work used to establish the reference separately. Do not assign a range to a failed or unfinished trial.
3. **Compare at the required accuracy.** Among the tested step sizes, identify the least-work run for each method with $|R-R_{\mathrm{ref}}|\leq0.009\ \mathrm{m}$, reserving the remaining 1 mm of the 1 cm target for the supported reference uncertainty. Require continued refinement to support the result, not a single accidentally close coarse-grid value. Compare the evaluation counts for the qualifying Euler and RK2 runs; they need not use the same time step.

Keep a compact results table with columns for **method, time-step size, range, difference from the reference, right-hand-side evaluations, and status**. Plotting range error against RHS evaluations will show both accuracy at comparable work and the least-work run that crosses the required threshold. A same-$\Delta t$ comparison remains useful supporting evidence, but it is not the final efficiency criterion.

Your completed work should contain the midpoint and steady-glide derivations, the verified touchdown procedure, the reference-refinement evidence, the results table, and a short conclusion naming which method required less work for the supported accuracy. If the evidence is insufficient, say so. The fixed-time convergence study is useful method evidence, but it does not stand in for touchdown-range convergence.

Once your Stage 1 evidence is complete, continue to the bounded investigation below. It reuses the verified evaluator and comparison procedure, while keeping time-step refinement separate from refinement of the launch grid.

(paper-airplane-agent-search)=
## With an agent: investigate launches

We return to the motivating question: which tested launch travels farthest, and which method supports that predicted range with less work? A grid search is repetitive, but its numerical rules are no longer mysterious. You have already derived the midpoint method, reconstructed the event logic, verified touchdown interpolation, and established the controlled accuracy-and-work procedure. The refinement study supplies an evidence-based **starting** step for the search; it does not assume that every launch has the same numerical error. The shortlist will be refined again.

In this activity, the AI agent may **write code into your learner-owned notebook, but it must not run any code**. You will inspect each proposed cell, execute it yourself, and decide what its output supports. This is your first direct notebook-editing task with an agent, so the permissions and stopping points are intentionally narrow.

### Comparison with engineering practice

We wrote the touchdown checks by hand because seeing them makes status, event location, and work accounting understandable. Engineers do not usually rewrite every defensive check for every study. They may reuse a tested solver or project utility, and a coding agent may draft an experiment driver. The engineer still specifies valid inputs and outcomes, inspects the generated code, verifies representative cases, and owns the conclusion. Here, the verified evaluator is protected; only the repetitive experiment code is delegated.


### Prepare the supplied task brief

Continue in the notebook where you reconstructed the lesson up to here. Save a checkpoint or download a backup before granting edit access. Then copy the brief below into a Markdown cell and replace only the three bracketed fields. The main numerical choices are supplied; you are not expected to design a full agent specification from scratch yet.

#### Agent launch-search task brief

**Your record**

- Notebook and checkpoint: `[enter the notebook filename and backup/checkpoint name]`
- Search step: `dt_search =` `[enter the finest accepted RK2 step from your reference evidence]`. My evidence for this choice is: `[one sentence referring to the two successive range changes]`
- Provenance: `[enter the date and the agent info, e.g. the persona name and model displayed by Jupyter AI]`

**Outcome**

Add two readable, unexecuted Python cells immediately below this brief. The first performs a bounded coarse search. The second refines time step and launch sampling separately. Neither cell chooses a final launch or writes the verdict.

**Use the notebook's existing work**

Use `np`, `rk2_step()`, `rhs_full_phugoid()`, `integrate_until_touchdown()`, the current model parameters, release point, and 15 s time limit. Do not redefine or modify the model, step methods, or touchdown evaluator. Keep angles in degrees when reporting them and convert to radians only when constructing a state. Use straightforward loops and comments; do not add a new package.

**Cell 1 — coarse bounded search**

- Use 17 equally spaced speeds from 4 to 12 m/s, inclusive, and 25 equally spaced angles from $-30^\circ$ to $30^\circ$, inclusive. These give spacings of 0.5 m/s and $2.5^\circ$.
- Evaluate every pair with RK2 at `dt_search`. Keep one result record per pair, including speed, angle in degrees, range, status, and number of steps. A failed or unfinished run keeps `range=None` and is not ranked.
- Sort only successful runs by decreasing range. Display the five leading candidates and report how many runs had each status. Preserve the complete result list for Cell 2.

**Cell 2 — two distinct refinements**

1. First hold the five coarse launch pairs fixed and repeat them at `dt_search/2` and `dt_search/4`. Display the three ranges together so the learner can judge time-step sensitivity without changing launch sampling.
2. Next take the three leading successful candidates at the finest of those time steps. Around each one, test speeds offset by $-0.50,-0.25,0,0.25,0.50$ m/s and angles offset by $-2.5^\circ,-1.25^\circ,0,1.25^\circ,2.5^\circ$. Clip to the original bounds and avoid duplicate pairs. Use `dt_search/4`, display the five leading local candidates, and keep every status.
3. Report results only. Do not label a global optimum, select the learner's candidate, compare Euler with RK2, or write a verdict.

**Permissions and stopping rule**

You may inspect this attached notebook and add exactly the two requested cells below this brief. Do not execute any cell, alter an existing cell, create or download files, access the network, or install anything. Stop after the two cells are inserted and give a short change record naming the variables each new cell creates.


:::{note} Python refresher — parameter grids and ranked records
:icon: false
`np.linspace(start, stop, count)` creates a requested number of evenly spaced values and includes both endpoints. Two nested `for` loops visit every speed–angle pair: the outer loop chooses one speed, and the inner loop tries all angles at that speed. `np.deg2rad(angle_degrees)` converts an angle before it enters the model.

A convenient search record is a dictionary such as `{'speed': ..., 'angle_deg': ..., 'range': ..., 'status': ...}`; a list holds all those records. Code may rank successful records with a line like `successful.sort(key=lambda row: row['range'], reverse=True)`. Here, `lambda row: row['range']` is a small unnamed function that supplies the sorting value, and `reverse=True` puts the largest range first. If the agent uses a more compact pattern you do not recognize, ask it to rewrite that pattern with ordinary loops and comments before you run it.
:::

### Ask for the two search cells

Select the agent you will work with (e.g., Jupyter AI with the Codex persona), add your notebook with the completed task brief, and send this invocation:

:::{card} Prompt
Follow the Agent launch-search task brief strictly. Inspect the existing notebook definitions, then add exactly the two requested Python cells below the brief. Keep the code readable for a beginning Python learner. Do not execute any cell and do not change existing cells. When the cells are inserted, stop and give me the requested change record.
:::

The prompt is short because the attached brief carries the numerical specification, permissions, and completion conditions. A long conversational prompt is not a substitute for that written record.

### Audit first, then execute one cell at a time

Do not begin with **Run All**. Read the agent's change record, inspect the notebook diff if the agent provides one, and check the first proposed cell line by line. Confirm that it:

- left all earlier cells unchanged and has no execution count or output;
- uses exactly the stated bounds, includes their endpoints, and converts degrees to radians;
- calls the existing RK2 step and touchdown evaluator rather than replacing them;
- preserves `status`, leaves failed ranges as `None`, and ranks only touchdown results;
- retains five candidates rather than trusting one coarse winner; and
- performs no file, network, installation, or execution operation.

When every item passes, execute it **yourself**. Inspect the status counts and five candidates. A grid full of failures, a candidate outside the bounds, or a suspiciously identical table is a reason to stop and diagnose, not to continue.

Next, audit the second agent-generated cell. Confirm that its first block changes only the time step and its second block changes only the launch sampling after choosing the refined time step. Then execute it yourself. Compare the three ranges for each fixed candidate. If two successive halvings do not change each leading range by less than 1 mm, or if the leading group is still changing materially, add another time-step halving before accepting the local search. If the best local result lies on the edge of every local neighborhood, note that the launch sampling may need another refinement.

If a cell raises an error, copy the traceback to the agent and permit it to revise **only the cell it wrote**. Inspect the revision and run it yourself again. Record the correction in your provenance note.

### Record a competitive candidate

Choose a candidate from the refined leading group; do not simply accept the first printed row. In a Markdown cell, record its speed, angle, refined RK2 range, nearby competitors, and the evidence that both the time step and local launch sampling were examined. Describe it as **competitive among the tested alternatives**, not as the globally optimal launch.

Then create a small Python cell with the following pattern, replace both blanks with your selected values, and execute it yourself:

```python
v_candidate = _____              # m/s
theta_candidate_deg = _____      # degrees
theta_candidate = np.deg2rad(theta_candidate_deg)
u_candidate = np.array([v_candidate, theta_candidate, x_0, y_0])
```

### Ask for the candidate accuracy-and-work cell

The coarse search used RK2 to find promising launches. It did not answer which method reaches the required accuracy with less work. Add the second brief below as a Markdown cell under your executed `u_candidate` cell; it follows the same format as the first.

#### Agent candidate-comparison task brief

**Outcome**

Add exactly one readable, unexecuted Python cell immediately below this brief. It reuses the Stage 1 comparison procedure and the verified `integrate_until_touchdown()` function for the launch stored in `u_candidate`.

**Reference**

- Establish a fresh RK2 range reference for this candidate, beginning at `dt_search/4` and accepted when two successive step halvings change the range by less than 0.001 m. Do not reuse the baseline launch's reference value.
- Limit reference refinement to six further halvings and report insufficient evidence if the rule is not met.

**Comparison**

- Reuse the Stage 1 Euler and RK2 step-size ladder, starting at 0.1 s and extending it by no more than six halvings if needed, to support the 0.009 m difference-from-reference criterion and one further refinement.
- Count work as completed steps multiplied by `rhs_calls_per_step` for each method.
- Print a table with method, `dt`, interpolated range, absolute difference from the reference, RHS evaluations, and status. Report reference-generation work separately.

**Permissions and stopping rule**

Do not modify existing cells, run code, choose a winning method, or write my verdict. Stop after inserting the cell and summarize what it will compute.


Then send this invocation:

:::{card} Prompt
Follow the Agent candidate-comparison task brief immediately above. Add exactly the one requested Python cell below it, do not execute it, and stop with the requested summary.
:::

Audit this cell before execution. Check the candidate state, the shared event treatment, the 1 mm reference evidence, the 0.009 m comparison allowance, continued refinement, status handling, and work counts. Then execute the cell yourself. If either method has no qualifying tested run, refine further or report that the current evidence is insufficient.


:::{warning .simple .dropdown icon=false open=false} Your verdict
Write a short engineering conclusion in your notebook that answers the original challenge. Name the competitive launch and its supported range, cite the closest tested alternatives, and describe the separate time-step and launch-grid refinement evidence. For Euler and RK2, name the least-work tested run that meets the accuracy rule and compare their RHS-evaluation counts. End with the limitation that this is a result within the model and tested search, not a proof of a real-airplane or global optimum.

Also include your provenance record: the agent/persona, date, cells it added, any revision after an error, and the fact that you inspected and executed the cells. The numerical output is evidence; the verdict is yours.
:::

### Completion rubric

Use this short **Complete/Revise** rubric. The activity is complete when the following are visible in your notebook:

- a search that stays within the stated bounds and preserves all completion statuses;
- several retained candidates, followed by separate time-step and local launch-grid refinement;
- your pre-execution audit and record that you ran each proposed cell yourself;
- a candidate-specific reference and Euler–RK2 accuracy-and-work table; and
- a provenance note and defended verdict with an honest optimality limitation.